# Adaptador de datos — cohortes de nacimiento NCHS

Etapa del framework: adaptador de datos (E2). Lee los archivos *Period/Cohort Linked Birth/Infant Death* directamente de los ZIP en `../NCHS_data/raw/` y arma cada cohorte de nacimiento 2016–2023 como indica la guía de usuario: nacimientos del año N unidos con las muertes de los años N y N+1 de los nacidos en N, por número de secuencia de cohorte y año de muerte. Escribe un Parquet por cohorte en `data/interim/` (no se versiona) con los códigos originales y las etiquetas. Versión en inglés: `01_build_cohorts.ipynb` (mismo código).

Requisitos: `pandas`, `numpy` y `pyarrow` (`%pip install pyarrow` si falta). Cada cohorte tarda unos 2 minutos y usa unos 3 GB de memoria.

In [2]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from libraries.nchs_reader import build_cohort, COHORTS, OUT_DIR

REBUILD = False      # True: rebuild every cohort even if its Parquet file exists
OUT_FILE = ROOT / "data" / "outputs_01_build_cohorts.txt"

## 1. Construir las cohortes (se omiten las que ya existen)

In [3]:
summaries = []
for y in COHORTS:
    if REBUILD or not (OUT_DIR / f"cohort_{y}.parquet").exists():
        summaries.append(build_cohort(y))          # about 2 minutes and 3 GB of memory per cohort
    else:
        print(f"cohort {y}: already built")

cohort 2016: already built
cohort 2017: already built
cohort 2018: already built
cohort 2019: already built
cohort 2020: already built
cohort 2021: already built
cohort 2022: already built
cohort 2023: already built


## 2. Conteos agregados por cohorte

Población: nacidos vivos de madres residentes en EE. UU., de 22 a 47 semanas. Salida: `data/outputs_01_build_cohorts.txt`.

In [4]:
rows = []
for y in COHORTS:
    d = pd.read_parquet(OUT_DIR / f"cohort_{y}.parquet",
                        columns=["RESTATUS", "COMBGEST", "CO_SEQNUM", "died_infant", "label_main",
                                 "label_early_neonatal", "died_first_24h", "label_day1_27"])
    us = d["RESTATUS"] < 4
    ga = d["COMBGEST"]
    pop = us & ga.between(22, 47).fillna(False)
    p28 = us & ga.between(28, 47).fillna(False)
    rows.append({
        "cohort": y, "births": len(d), "births_us_resident": int(us.sum()),
        "infant_deaths_linked": int(d["died_infant"].sum()),
        "ga_unknown": int((us & (ga == 99)).sum()), "ga_lt22": int((us & (ga < 22)).sum()),
        "pop_ge22wk": int(pop.sum()),
        "events_main_pop": int(d.loc[pop, "label_main"].sum()),
        "events_first24h_pop": int(d.loc[pop, "died_first_24h"].sum()),
        "events_day1_27_pop": int(d.loc[pop, "label_day1_27"].sum()),
        "events_early_neonatal_pop": int(d.loc[pop, "label_early_neonatal"].sum()),
        "pop_ge28wk": int(p28.sum()), "events_main_ge28": int(d.loc[p28, "label_main"].sum()),
    })
t = pd.DataFrame(rows).set_index("cohort").T
lines = ["Data adapter: birth cohorts built from the NCHS Period/Cohort Linked Birth/Infant Death "
         "public-use files (aggregate counts only)",
         "pop = live births to US-resident mothers with combined gestation 22-47 weeks; "
         "main label = neonatal death at 0-27 days", ""] + t.to_string().splitlines()
OUT_FILE.write_text("\n".join(lines) + "\n", encoding="utf-8")
t

cohort,2016,2017,2018,2019,2020,2021,2022,2023
births,3956112,3864754,3801534,3757582,3619826,3669928,3676029,3605081
births_us_resident,3945875,3855500,3791712,3747540,3613647,3664292,3667758,3596017
infant_deaths_linked,22893,22167,21346,20639,19346,19965,20268,19743
ga_unknown,3457,2760,2509,2407,2478,2993,2671,2642
ga_lt22,4130,4182,3960,3946,3690,3537,3563,3549
pop_ge22wk,3938288,3848558,3785243,3741187,3607479,3657762,3661524,3589826
events_main_pop,11497,11057,10703,10249,9574,9633,9967,9879
events_first24h_pop,5107,5356,5161,5136,4823,4556,4697,4620
events_day1_27_pop,6390,5701,5542,5113,4751,5077,5270,5259
events_early_neonatal_pop,8605,8222,7998,7605,7110,6988,7148,7089
